# 遠征前壓力測試：什麼樣的資料會讓固定的 Flow Matching framework 開始做不好？

課堂實作裡，同一個 `FlowMatching` class 可以生成影像，也可以生成 action chunks。看起來只要換資料與 neural network 的 input/output shape，事情就解決了。

但真實世界的資料不會永遠像課堂 toy 那麼友善：有些 modes 很少、有些答案擠得很近、有些成功路徑非常窄，有些 observations 則沒有把真正重要的資訊全部說出來。

所以這份作業要問的是：

> **在不修改 Flow Matching 演算法的前提下，什麼樣合理的 continuous data problem，會讓課堂上的固定 framework 開始做不好？我們怎麼知道原因是什麼？**

這是一份兩週作業。兩週內請自己安排時間；**沒有中間 checkpoint**。最後一次繳交：

1. 完成作答與實驗的 `.ipynb`；
2. 同一份 notebook 的 PDF 或 HTML export；
3. 一張 summary figure；
4. AI／research decision log。

這份作業不要求你把模型救回來。沒有成功讓它完全失效，但問題合理、預測清楚、實驗能推翻自己、分析誠實，會比做出很大的 failure 卻說不清楚原因得到更高的分數。


## 評分方式

| 項目 | 配分 |
| --- | ---: |
| 真實問題與 toy 的合理對應 | 20 |
| 數學定義、notation 與 symbol-to-code | 20 |
| Failure intuition 與可被推翻的預測 | 15 |
| 控制變因、結果與 useful value | 25 |
| 結果符合／不符合預期的解釋 | 15 |
| Reproducibility 與 AI-use appendix | 5 |

數字好不好只占「結果與 useful value」的一部分。你真正要交的是一條可以被檢查的推理鏈：

$$
\text{real problem}
\rightarrow \text{controlled toy}
\rightarrow \text{failure intuition}
\rightarrow \text{prediction}
\rightarrow \text{experiment}
\rightarrow \text{explanation}.
$$


---

# 0. 先把不能改的東西固定下來

這份作業的自由度在 **data problem**，不在 Flow Matching algorithm。所有正式實驗都固定：

$$
T\sim U[0,1],
\qquad
X_0\sim\mathcal N(0,I),
\qquad
(X_1,C)\sim p_{\mathrm{data}}.
$$

$X_0$ 與 $(X_1,C)$ 獨立抽樣；$X_1$ 與 $C$ 則必須來自同一筆資料。不能把某一筆 observation 和另一筆資料的 target 隨便配在一起。

Probability path 與 training target 固定為

$$
I_t=(1-t)X_0+tX_1,
\qquad
U_t=X_1-X_0.
$$

Training objective 固定為

$$
\mathcal L(\theta)
=
\mathbb E_{T,X_0,(X_1,C)}
\left[
\left\|v_\theta(I_T,T,C)-U_T\right\|^2
\right].
$$

Sampling equation 固定為

$$
\frac{\mathrm dX_t}{\mathrm dt}
=v_\theta(X_t,t,c),
\qquad
X_0\sim\mathcal N(0,I).
$$

如果是 unconditional problem，請令 `C = None`；不要另外發明一個沒有作用的 condition。


## 正式實驗的固定設定

你可以因為資料 shape 改變 network 的 input/output dimension 與 `cond_dim`，但不能改下面的 capacity 與 training/sampling budget。

| 設定 | 固定值 |
| --- | ---: |
| Velocity network | `FlatVelocityNetwork` |
| Hidden width | 256 |
| Hidden layers | 3 |
| Batch size | 256 |
| Optimizer | Adam |
| Learning rate | $2\times 10^{-3}$ |
| Training steps | 3,000 |
| Sampler | Heun |
| ODE integration steps | 30 |
| Network evaluations | 60（每個 Heun step 兩次） |
| Final evaluation samples | 1,000 |
| Final training seeds | 101, 202, 303 |

你可以用更少的 steps 與較少 samples 做 smoke test，但不能把 smoke-test 數字當成正式結果。


In [ ]:
import math, time
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

HIDDEN = 256
LAYERS = 3
BATCH_SIZE = 256
LEARNING_RATE = 2e-3
TRAIN_STEPS = 3000
SAMPLE_STEPS = 30
N_EVAL = 1000
FINAL_SEEDS = [101, 202, 303]

def set_seed(seed=0):
    np.random.seed(seed)
    torch.manual_seed(seed)

print("device:", DEVICE)


## 共用的 Flow Matching code

下面兩個 classes 和課堂實作相同。正式實驗不得修改。

`FlatVelocityNetwork` 是我們刻意固定的簡單 baseline。它不一定是影像或軌跡最好的 architecture；正因為它固定，才有辦法比較不同 data properties 對同一個 framework 的影響。


In [ ]:
def time_features(t, n_freq=4):
    t = t.reshape(-1, 1)
    k = (2 ** torch.arange(n_freq, device=t.device, dtype=t.dtype)) * math.pi
    return torch.cat([t, torch.sin(t * k), torch.cos(t * k)], dim=1)


class FlatVelocityNetwork(nn.Module):
    def __init__(self, sample_shape, cond_dim=0, hidden=HIDDEN, layers=LAYERS, n_freq=4):
        super().__init__()
        self.sample_shape = tuple(sample_shape)
        self.cond_dim = cond_dim
        self.n_freq = n_freq
        data_dim = math.prod(self.sample_shape)
        in_dim = data_dim + (1 + 2 * n_freq) + cond_dim

        blocks = []
        for _ in range(layers):
            blocks += [nn.Linear(in_dim, hidden), nn.SiLU()]
            in_dim = hidden
        blocks.append(nn.Linear(in_dim, data_dim))
        self.net = nn.Sequential(*blocks)

    def forward(self, x, t, c=None):
        batch = len(x)
        pieces = [x.reshape(batch, -1), time_features(t, self.n_freq)]
        if self.cond_dim:
            if c is None:
                raise ValueError("This velocity network requires condition C.")
            pieces.append(c.reshape(batch, self.cond_dim))
        velocity = self.net(torch.cat(pieces, dim=1))
        return velocity.reshape_as(x)


class FlowMatching:
    def __init__(self, velocity, lr=LEARNING_RATE):
        self.velocity = velocity
        self.lr = lr

    def loss(self, x0, x1, c=None):
        batch = len(x0)
        t = torch.rand(batch, device=x0.device)
        t_view = t.reshape(batch, *([1] * (x0.ndim - 1)))
        xt = (1 - t_view) * x0 + t_view * x1
        target_velocity = x1 - x0
        return ((self.velocity(xt, t, c) - target_velocity) ** 2).mean()

    def fit(self, get_batch, steps=TRAIN_STEPS, name="model", verbose=True):
        optimizer = torch.optim.Adam(self.velocity.parameters(), lr=self.lr)
        history = []
        report_every = max(1, steps // 5)
        for step in range(1, steps + 1):
            loss = self.loss(*get_batch())
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()
            history.append(loss.item())
            if verbose and (step == 1 or step % report_every == 0):
                print(f"[{name}] step {step:4d}/{steps}: loss = {np.mean(history[-report_every:]):.4f}")
        return history

    @torch.no_grad()
    def sample(self, x0, c=None, steps=SAMPLE_STEPS):
        x = x0.clone()
        dt = 1.0 / steps
        batch = len(x)
        for i in range(steps):
            t0 = torch.full((batch,), i / steps, device=x.device)
            t1 = torch.full((batch,), (i + 1) / steps, device=x.device)
            v0 = self.velocity(x, t0, c)
            proposal = x + dt * v0
            v1 = self.velocity(proposal, t1, c)
            x = x + 0.5 * dt * (v0 + v1)
        return x


---

# 問題一：從真實世界設計一個受控制的 toy

先不要想怎麼讓模型爆掉。先找一個真實世界可能出現的 data property，再問：**怎麼把它縮成一個保留核心困難、但答案仍然看得見的 toy？**

你可以選擇影像生成，或動作與軌跡 policy learning。只能選一個 primary data property。

## 合理的設計模式

| 影像資料 | 軌跡 policy 資料 |
| --- | --- |
| Long-tailed modes | Narrow success region |
| Modes 很接近或重疊 | 同一 state 有多種策略 |
| 不同 modes 有不同 variation | Demonstration imbalance |
| 合法性取決於 global pixel structure | 不同 states 有不同難度 |
| 簡化的 condition 對應多種影像 | Partial observation |
| 結構複雜度隨參數增加 | 較長時間的 dependency |

這張表描述的是設計方向，不是完整題目。你仍然要說明它對應到哪一種真實資料，以及你的 toy 保留了什麼、拿掉了什麼。


## 不接受的 failure

- 把 continuous output 換成 discrete token 或 class index；
- 把 Gaussian variation 完全拿掉，使資料退化成純粹的有限離散點；
- 故意使用錯誤 normalization、錯誤 shape 或錯誤 condition pairing；
- 修改 path、training target、loss 或 sampler；
- 減少 training steps、network width 或 NFE；
- 只換一個比較差的 random seed；
- 同時改很多 data properties，最後無法知道是哪一項造成差異；
- 要求模型使用從來沒有放進 $C$ 的資訊，卻不承認這是 partial observability。

你的 $X_1$ 必須保持為 continuous real-valued tensor。Partial observation 可以研究，但必須把「模型能否表示 $p(X_1\mid C)$」和「$C$ 是否足以決定真實任務」分開討論。


## 1.1 真實世界原型

請先回答以下問題，再寫 toy generator：

1. 你選擇影像還是 trajectory policy？
2. 真實世界中，什麼資料會具有你選的 property？
3. 為什麼這個 property 可能讓 generative model 變得困難？
4. 請放一張 real-data figure、資料範例或自己畫的情境圖。

### 你的回答

**領域：** `[請填寫]`

**真實情境：**

`[請在這裡作答並插入圖片]`

**你想保留的 data property：**

`[請在這裡作答]`


## 1.2 Real problem 怎麼縮成 toy？

| 問題 | 你的設計 |
| --- | --- |
| 真實世界的一筆資料是什麼？ | `[請填寫]` |
| Toy 裡的 $X_1$ 是什麼？ | `[請填寫]` |
| Condition $C$ 是什麼？ | `[請填寫]` |
| Toy 保留哪一個困難？ | `[請填寫]` |
| Toy 刪除了哪些真實複雜度？ | `[請填寫]` |
| 為什麼刪掉後仍能回答你的問題？ | `[請填寫]` |

只能選一個 primary change。請用一句話完成：

> **相較於課堂 toy，我只改變 __________；其他資料設定保持不變。**


## 1.3 用數學定義你的資料

如果是 conditional problem，請寫出

$$
(X_1,C)\sim p_{\mathrm{toy},\lambda}(x_1,c).
$$

如果是 unconditional problem，請寫出

$$
X_1\sim p_{\mathrm{toy},\lambda}(x_1),
\qquad C=\varnothing.
$$

$\lambda$ 是你用來控制困難程度的參數。至少定義三個值：

$$
\lambda_{\mathrm{easy}},
\qquad
\lambda_{\mathrm{medium}},
\qquad
\lambda_{\mathrm{hard}}.
$$

請完整說明資料的 sampling procedure。不要只寫「從我的 dataset 抽一筆」。

### 你的數學定義

`[請在這裡寫下 distribution、sampling procedure，以及三個 difficulty levels]`


## 1.4 Symbol-to-code table

| 數學符號 | 在你的問題裡代表什麼？ | Tensor shape | 程式變數 |
| --- | --- | --- | --- |
| $X_1$ | `[請填寫]` | `[請填寫]` | `x1` |
| $C$ | `[請填寫]` | `[請填寫]` | `c` |
| $X_0$ | Gaussian source | 與 $X_1$ 相同 | `x0` |
| $I_t$ | `[請填寫]` | `[請填寫]` | `xt` |
| $U_t$ | `[請填寫]` | `[請填寫]` | `target_velocity` |
| $\lambda$ | `[請填寫]` | scalar | `difficulty` |

接著用一句話解釋：為什麼 $X_1$ 與 $C$ 必須一起抽，而 $X_0$ 可以獨立抽？

`[請在這裡作答]`


## 1.5 實作你的 toy data generator

完成下面三個設定與 `sample_target`。它必須回傳：

- `x1`：shape `(batch, *DATA_SHAPE)`；
- `c`：shape `(batch, COND_DIM)`，或 unconditional 時回傳 `None`。

同一個 `difficulty` 必須代表同一個資料分布；不要在 easy、medium、hard 之間偷偷改第二個 property。


In [ ]:
# TODO: choose exactly one track.
TRACK = "image"  # "image" or "trajectory"

# TODO: replace these with your data shapes.
DATA_SHAPE = (1, 8, 8)
COND_DIM = 0

# TODO: define the three values mathematically before filling them here.
DIFFICULTY_LEVELS = {
    "easy": 0.0,
    "medium": 0.5,
    "hard": 1.0,
}

def sample_target(batch_size, difficulty, device=DEVICE):
    # Return paired samples (x1, c) from p_toy,difficulty.
    # x1: (batch_size, *DATA_SHAPE), float32
    # c:  (batch_size, COND_DIM), float32, or None
    # TODO: implement your continuous toy distribution.
    raise NotImplementedError


## 1.6 先只看資料，不要先看 model results

對 easy、medium、hard 各畫至少 16 筆 $X_1$。如果是 conditional problem，圖上也要標出對應的 $C$。

看完後回答：

- $\lambda$ 增加時，肉眼可以看到什麼變化？
- 哪些東西刻意保持相同？
- 這三組資料真的只差一個 primary property 嗎？


In [ ]:
# TODO: visualize at least 16 data samples from each difficulty level.
# Your figure should make the controlled change visible without showing model outputs.

fig = None
# ... your visualization code ...


### 你從資料圖讀到什麼？

`[請在這裡作答。若資料圖已經和你原先想像不同，現在就要說明。]`


---

# 問題二：為什麼你覺得固定的 Flow Matching 會做不好？

先別跑正式訓練。

你現在已經定義好資料。接著要先留下自己的 failure intuition：不是「因為 hard 比較難」，而是具體指出 **velocity network 在什麼地方要同時處理哪些互相衝突、稀少或精細的訊號**。


## 2.1 先用直覺解釋

請回答：

1. $\lambda$ 從 easy 走向 hard 時，資料幾何或 conditional structure 發生什麼變化？
2. 對相近的 $(I_t,t,C)$，可能出現哪些不同的 target velocities？
3. 這些 velocities 是方向接近、互相抵消，還是只有少數樣本才看得到？
4. 你預期 generated samples 會怎麼壞？
5. 哪一群 samples、哪一種 condition，或哪一段 $t$ 應該最先壞？

請加一張自己畫的示意圖。圖中至少標出 $X_0$、$X_1$、$I_t$ 與兩個可能的 velocity targets。

### 你的直覺

`[請在這裡作答並插入示意圖]`


## 2.2 用一條式子把直覺釘住

Squared-error regression 的最佳解是

$$
v^\star(x,t,c)
=
\mathbb E[X_1-X_0\mid I_t=x,C=c].
$$

請用你的 toy 回答：

- 哪些不同的 $(X_0,X_1)$ 可能產生相近的 $(I_t,C)$？
- 它們的 $X_1-X_0$ 如何不同？
- 取 conditional mean 後，哪一種資訊可能被保留、變弱或抵消？

不要求完整求出 $v^\star$，但你的直覺必須能和這個 conditional expectation 對上。

### 你的回答

`[請在這裡作答]`


## 2.3 寫下可被推翻的預測

這一節必須放在 results 前面。即使實驗結果不同，也不要刪掉或改寫成事後看來正確的版本。

| 問題 | 實驗前的預測 |
| --- | --- |
| 你定義的 failure 是什麼？ | `[請填寫 quantitative criterion]` |
| 哪個 metric 會最先變差？ | `[請填寫]` |
| easy → medium → hard 應該是漸進還是突然惡化？ | `[請填寫]` |
| 哪類 samples／conditions 最先失敗？ | `[請填寫]` |
| 什麼結果會推翻你的 mechanism？ | `[請填寫]` |

再用一個完整句子押注：

> **我預測當 $\lambda$ 從 ______ 增加到 ______ 時，______ 會先發生，因為 ______。**


## 2.4 定義 metrics 與一個 useful value

至少使用一個原有 task metric：

- 影像：valid rate、mode coverage、per-mode recall、nearest-template distance；
- 軌跡：collision rate、success rate、route coverage、goal error。

另外自行定義一個直接服務於 explanation 的 useful value，例如：

- rare mode 在 training 中預期出現的次數；
- mode separation／within-mode standard deviation；
- conditional mean image 到最近合法模板的距離；
- mean action trajectory 到障礙物的最小距離；
- 相近 $(I_t,C)$ 的 target-velocity disagreement；
- 各 time bins 的 velocity error。

先寫公式、說明它為什麼有用，再寫 code。不要先印出一個容易算的數字，才替它補理由。

### 你的 metric 與 useful value

`[請在這裡寫公式、定義與預測方向]`


In [ ]:
def evaluate_samples(generated, reference, condition, difficulty):
    # Return a dictionary of scalar metrics.
    # Include at least one task metric and your useful value (or its sample-level quantity).
    # TODO: implement your metrics.
    raise NotImplementedError


def compute_useful_value(difficulty):
    # Compute the diagnostic quantity defined in your report.
    # TODO: this may use data samples without training a model.
    raise NotImplementedError


---

# 3. 正式實驗

現在才開始訓練。

Easy、medium、hard 必須使用完全相同的 model capacity、training budget、sampler、NFE 與 evaluation sample count。三個正式 seeds 也必須相同，這樣每一組才是 paired comparison。

下面的 experiment runner 不需要知道你選影像還是軌跡。它只依賴前面定義的 `DATA_SHAPE`、`COND_DIM`、`sample_target` 與 `evaluate_samples`。


In [ ]:
def checked_target(batch_size, difficulty):
    x1, c = sample_target(batch_size, difficulty, device=DEVICE)
    x1 = x1.to(device=DEVICE, dtype=torch.float32)
    assert tuple(x1.shape[1:]) == tuple(DATA_SHAPE)
    if COND_DIM == 0:
        assert c is None
    else:
        assert c is not None
        c = c.to(device=DEVICE, dtype=torch.float32)
        assert tuple(c.shape) == (batch_size, COND_DIM)
    return x1, c


def train_one(difficulty, seed, steps=TRAIN_STEPS, verbose=True):
    set_seed(seed)
    velocity = FlatVelocityNetwork(
        sample_shape=DATA_SHAPE,
        cond_dim=COND_DIM,
        hidden=HIDDEN,
        layers=LAYERS,
    ).to(DEVICE)
    flow = FlowMatching(velocity, lr=LEARNING_RATE)

    def get_batch():
        x1, c = checked_target(BATCH_SIZE, difficulty)
        x0 = torch.randn_like(x1)
        return x0, x1, c

    history = flow.fit(
        get_batch,
        steps=steps,
        name=f"lambda={difficulty}, seed={seed}",
        verbose=verbose,
    )
    return flow, history


@torch.no_grad()
def generate_and_evaluate(flow, difficulty, seed, n=N_EVAL, sample_steps=SAMPLE_STEPS):
    set_seed(seed + 10_000)
    reference, c = checked_target(n, difficulty)
    x0 = torch.randn_like(reference)
    generated = flow.sample(x0, c=c, steps=sample_steps)
    metrics = evaluate_samples(generated, reference, c, difficulty)
    return generated.cpu(), reference.cpu(), None if c is None else c.cpu(), metrics


## 3.1 Smoke test

先用 20 training steps、32 evaluation samples 檢查 shapes 與 code path。這裡的數字沒有科學意義，不得放進正式結論。


In [ ]:
SMOKE_DIFFICULTY = DIFFICULTY_LEVELS["easy"]

# Uncomment after sample_target() and evaluate_samples() are implemented.
# smoke_flow, _ = train_one(SMOKE_DIFFICULTY, seed=0, steps=20)
# smoke_generated, smoke_reference, smoke_c, smoke_metrics = generate_and_evaluate(
#     smoke_flow, SMOKE_DIFFICULTY, seed=0, n=32, sample_steps=2
# )
# print(smoke_metrics)


## 3.2 三個 difficulty levels × 三個 paired seeds

這會訓練九個 models。確認 smoke test 正常後才把 `RUN_FINAL` 改成 `True`。


In [ ]:
RUN_FINAL = False

records = []
example_outputs = {}

if RUN_FINAL:
    for level_name, difficulty in DIFFICULTY_LEVELS.items():
        print(f"\n===== {level_name}: lambda={difficulty} =====")
        print("useful value:", compute_useful_value(difficulty))
        for seed in FINAL_SEEDS:
            flow, history = train_one(difficulty, seed=seed)
            generated, reference, c, metrics = generate_and_evaluate(
                flow, difficulty, seed=seed
            )
            record = {"level": level_name, "difficulty": difficulty, "seed": seed, **metrics}
            records.append(record)
            print(record)
            if seed == FINAL_SEEDS[0]:
                example_outputs[level_name] = {
                    "generated": generated,
                    "reference": reference,
                    "condition": c,
                    "history": history,
                }
            del flow
            if torch.cuda.is_available():
                torch.cuda.empty_cache()


## 3.3 Summary table 與主圖

Summary table 至少要有每個 metric 的 mean ± standard deviation。主圖必須讓讀者同時看到：

1. easy／medium／hard 的資料差異；
2. easy／medium／hard 的生成結果；
3. 至少一個 metric 隨 $\lambda$ 的變化；
4. 你定義的 useful value。

圖不能只放最好看的 seed。範例視覺可以使用固定 seed，但數字必須來自三個 seeds。


In [ ]:
# TODO: aggregate records into mean ± std and create the summary figure.
# You may use pandas if you want, but it is not required.

# ... summary table ...
# ... one summary figure ...


### Results：圖和數字實際說了什麼？

先描述 observation，不要在第一句就宣稱原因。

`[請在這裡放 summary figure、summary table，並描述最重要的兩到三個 observations。]`


---

# 4. 結果和原本想的一樣嗎？

回到 2.3 的押注，逐項比較。不要只寫「符合預期」或「不符合預期」。

| 原本的預測 | 實際結果 | 符合程度 | 支持你的證據 |
| --- | --- | --- | --- |
| `[請填寫]` | `[請填寫]` | 符合／部分符合／不符合 | `[figure、metric 或 useful value]` |
| `[請填寫]` | `[請填寫]` | 符合／部分符合／不符合 | `[figure、metric 或 useful value]` |


## 4.1 如果結果符合預期

請回答：

- 哪一張圖或哪一個數字最直接支持原本的 failure mechanism？
- $v^\star(x,t,c)=\mathbb E[X_1-X_0\mid I_t=x,C=c]$ 如何幫助解釋？
- 你的 useful value 能不能預測 failure 發生的位置或強度？
- 是否還有另一個 mechanism 也可能產生相同結果？

## 4.2 如果結果不符合預期

請回答：

- 模型真的沒有失敗，還是 metric 沒有抓到 failure？
- Toy 是否沒有保留你以為的真實困難？
- Flow Matching 是否用你沒有預期到的方式保留了 distribution？
- 哪一個額外計算可以區分這些解釋？

你可以追加一個 revised setting，但必須保留原本的 negative result。

### 你的解釋

`[請依照符合／不符合的情況作答。請把 intuition、公式、圖與 useful value 接成同一條推理。]`


## 4.3 一個 diagnostic experiment

最後可以做一個診斷實驗，目的不是把結果救到最好，而是區分原因。只能改一件事，例如：

- hard setting 增加 NFE；
- hard setting 增加 training steps；
- hard setting 暫時增加 network width；
- partial-observation toy 補回缺少的 condition；
- imbalanced toy 暫時重新平衡資料。

Diagnostic 不算正式方法，因此必須和前面的固定設定分開報告。

先寫下：**如果你的 mechanism 是對的，這個 diagnostic 應該改變什麼？**

`[請先作答，再執行下面的實驗]`


In [ ]:
# TODO: run exactly one diagnostic intervention on the hard setting.
# Keep the original hard-setting result. Do not replace it with this diagnostic result.

# ... diagnostic experiment ...


### Diagnostic 結果

`[它支持、削弱，還是無法區分你的 explanation？為什麼？]`


---

# 5. AI／research decision log

你可以使用 AI 協助 coding、debug、公式檢查、視覺化與文字整理。評分不看 AI 用得多或少，而是看你有沒有驗證建議，以及能不能解釋最後留下來的內容。

如果沒有使用 AI，也請記錄三個自己或與同學討論後做出的重要研究決策。

| 你當時遇到的問題 | 建議來自 AI／同學／自己 | 得到的建議 | 你如何驗證？ | 接受或拒絕？為什麼？ |
| --- | --- | --- | --- | --- |
| `[請填寫]` |  |  |  |  |
| `[請填寫]` |  |  |  |  |
| `[請填寫]` |  |  |  |  |

最後請列出哪些 code cells 曾受到 AI 協助，以及你實際檢查了什麼。不要只寫「AI 幫我 debug」。


# 6. 最後回到開頭

請用 150–250 字回答：

> **你的實驗告訴我們，固定的 Flow Matching framework 面對哪一種真實 data property 時會開始遇到困難？目前證據支持的原因是什麼？哪些部分仍然不能確定？**

`[請在這裡寫最後結論]`


## 繳交前 checklist

- [ ] 我的 toy 來自一個說得清楚的真實資料現象。
- [ ] $X_1$ 仍是 continuous real-valued tensor。
- [ ] Easy、medium、hard 只改一個 primary property。
- [ ] 我沒有修改 fixed Flow Matching path、target、loss 或 sampler。
- [ ] 我定義了 $p_{\mathrm{toy},\lambda}$、所有 symbols 與 tensor shapes。
- [ ] Failure intuition 寫在 results 前面，而且可以被實驗推翻。
- [ ] 正式結果使用三個 paired seeds。
- [ ] 我報告了 task metric 與自己定義的 useful value。
- [ ] Negative result 沒有被刪掉。
- [ ] Explanation 同時引用 intuition、公式、圖與數字。
- [ ] Notebook 可以 restart and run all。
- [ ] 我完成 AI／research decision log。
- [ ] 我已輸出 PDF 或 HTML，確認公式、圖與表格沒有被切掉。
